# ChatGPT Archive Compiler — semantic atlas and thematic book

> **Superseded for large archives:** this first revision purchases a structured profile for every conversation and can be prohibitively expensive. Use `02_semantic_atlas_budgeted_colab.ipynb`, which preserves complete local coverage while enforcing bounded representative analysis and a persistent API spending ceiling.

This third notebook transforms the validated Archive IR into a **semantic atlas**: a multi-label catalog of subjects, projects, conversational purposes, recurring themes, entities, relationships, and long-running intellectual threads. It then renders an organized, cross-referenced book whose primary structure is thematic rather than merely chronological.

The analytical implementation lives in the versioned Python package. This notebook is deliberately a thin, auditable orchestration layer: it creates an ephemeral checkout under **/content**, verifies the analysis planner with synthetic conversations, performs a no-network estimate on the real archive, and only then—after explicit authorization—runs the resumable analysis and book renderer.

The default **enhanced** mode uses OpenAI models for high-resolution interpretation after local preprocessing and batching. Expensive stages are cached in the commit-stamped Drive output directory. The notebook never prints titles, excerpts, category labels, entity names, identifiers, model responses, or source-derived exception text. Only aggregate counts, configuration identifiers, commit identifiers, and artifact paths are displayed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

## Configuration

The defaults favor analytical depth and a polished print object. `gpt-5.6` interprets cluster structure and composes controlled summaries; `text-embedding-3-large` supplies the semantic geometry. Both identifiers remain editable so a future model change does not require modifying package code. Structured stages also have explicit output-token ceilings, including reasoning tokens; preflight reports those ceilings separately from expected consumption.

The book renderer uses a deliberately restrained navy-and-copper editorial system, book serif text, sans-serif navigation, trade-book geometry, generous margins, strong part and chapter openings, running furniture, controlled line length, widow/orphan controls, and print-aware cross-references. It stores self-contained HTML alongside PDF so the atlas remains searchable and browsable. A global profile budget keeps the printed edition selective while every conversation remains listed in its category directory.

In [ ]:
from pathlib import Path

REPO_BRANCH = "main"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs" / "semantic"

ANALYSIS_MODE = "enhanced"  # @param ["enhanced", "local"]
ANALYSIS_MODEL = "gpt-5.6"  # @param {type:"string"}
EMBEDDING_MODEL = "text-embedding-3-large"  # @param {type:"string"}
EMBEDDING_DIMENSIONS = 1024  # @param {type:"integer"}
REASONING_EFFORT = "high"  # @param ["medium", "high"]
SYNTHESIS_REASONING_EFFORT = "xhigh"  # @param ["high", "xhigh"]
PROFILE_MAX_OUTPUT_TOKENS = 32000  # @param {type:"integer"}
TAXONOMY_MAX_OUTPUT_TOKENS = 64000  # @param {type:"integer"}
SYNTHESIS_MAX_OUTPUT_TOKENS = 120000  # @param {type:"integer"}
MAX_CHARACTERS_PER_CONVERSATION = 32000  # @param {type:"integer"}
EMBEDDING_BATCH_SIZE = 64  # @param {type:"integer"}
ANALYSIS_BATCH_SIZE = 8  # @param {type:"integer"}
NEAREST_NEIGHBORS = 12  # @param {type:"integer"}
MIN_SIMILARITY = 0.32  # @param {type:"number"}
MAX_LEAF_CATEGORIES = 64  # @param {type:"integer"}
REVIEW_CONFIDENCE_THRESHOLD = 0.65  # @param {type:"number"}
MAX_CONVERSATIONS = 0  # @param {type:"integer"}

# Editable planning snapshot; verify current prices before a materially later run.
PRICE_SNAPSHOT_DATE = "2026-07-19"
EMBEDDING_USD_PER_MILLION_INPUT_TOKENS = 0.13  # @param {type:"number"}
ANALYSIS_USD_PER_MILLION_INPUT_TOKENS = 5.00  # @param {type:"number"}
ANALYSIS_USD_PER_MILLION_OUTPUT_TOKENS = 30.00  # @param {type:"number"}
ESTIMATED_PROFILE_OUTPUT_TOKENS = 500  # @param {type:"integer"}
ESTIMATED_META_INPUT_TOKENS_PER_CONVERSATION = 1000  # @param {type:"integer"}
ESTIMATED_GLOBAL_OUTPUT_TOKENS = 100000  # @param {type:"integer"}

BOOK_TITLE = "A Semantic Atlas of My ChatGPT Archive"  # @param {type:"string"}
BOOK_SUBTITLE = (
    "Subjects, projects, connections, and intellectual trajectories"  # @param {type:"string"}
)
BOOK_AUTHOR = ""  # @param {type:"string"}
PAPER_SIZE = "trade"  # @param ["trade", "a4"]
MAX_EXPANDED_BOOK_PROFILES = 240  # @param {type:"integer"}
MAX_PROJECT_TIMELINES = 24  # @param {type:"integer"}
MAX_TIMELINE_EVENTS_PER_PROJECT = 12  # @param {type:"integer"}
RENDER_PDF = True  # @param {type:"boolean"}

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Drive folder is missing: MyDrive/ChatGPT Data Export")
if not REPO_BRANCH.strip():
    raise ValueError("REPO_BRANCH must not be empty.")
if ANALYSIS_MODE not in {"enhanced", "local"}:
    raise ValueError("ANALYSIS_MODE must be 'enhanced' or 'local'.")
if not ANALYSIS_MODEL.strip() or not EMBEDDING_MODEL.strip():
    raise ValueError("Model identifiers must not be empty.")
if REASONING_EFFORT not in {"medium", "high"}:
    raise ValueError("REASONING_EFFORT must be 'medium' or 'high'.")
if SYNTHESIS_REASONING_EFFORT not in {"high", "xhigh"}:
    raise ValueError("SYNTHESIS_REASONING_EFFORT must be 'high' or 'xhigh'.")
if (
    min(
        EMBEDDING_DIMENSIONS,
        MAX_CHARACTERS_PER_CONVERSATION,
        EMBEDDING_BATCH_SIZE,
        ANALYSIS_BATCH_SIZE,
        NEAREST_NEIGHBORS,
        MAX_LEAF_CATEGORIES,
        MAX_EXPANDED_BOOK_PROFILES,
        MAX_PROJECT_TIMELINES,
        MAX_TIMELINE_EVENTS_PER_PROJECT,
        PROFILE_MAX_OUTPUT_TOKENS,
        TAXONOMY_MAX_OUTPUT_TOKENS,
        SYNTHESIS_MAX_OUTPUT_TOKENS,
    )
    <= 0
):
    raise ValueError("Dimensions, limits, and batch sizes must be positive.")
if MAX_LEAF_CATEGORIES > 512:
    raise ValueError("MAX_LEAF_CATEGORIES must not exceed 512.")
if not -1.0 <= MIN_SIMILARITY <= 1.0:
    raise ValueError("MIN_SIMILARITY must be between -1 and 1.")
if not 0.0 <= REVIEW_CONFIDENCE_THRESHOLD <= 1.0:
    raise ValueError("REVIEW_CONFIDENCE_THRESHOLD must be between zero and one.")
if MAX_CONVERSATIONS < 0:
    raise ValueError("MAX_CONVERSATIONS must be zero or positive.")
if (
    min(
        EMBEDDING_USD_PER_MILLION_INPUT_TOKENS,
        ANALYSIS_USD_PER_MILLION_INPUT_TOKENS,
        ANALYSIS_USD_PER_MILLION_OUTPUT_TOKENS,
        ESTIMATED_PROFILE_OUTPUT_TOKENS,
        ESTIMATED_META_INPUT_TOKENS_PER_CONVERSATION,
        ESTIMATED_GLOBAL_OUTPUT_TOKENS,
    )
    < 0
):
    raise ValueError("Price and token-planning inputs must not be negative.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository branch: {REPO_BRANCH}")
print(f"Analysis mode: {ANALYSIS_MODE}")
print(f"Analysis model: {ANALYSIS_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL} ({EMBEDDING_DIMENSIONS} dimensions)")
print(f"Maximum leaf categories: {MAX_LEAF_CATEGORIES}")
print(
    f"Book edition: semantic editorial; paper: {PAPER_SIZE}; "
    f"expanded profiles: {MAX_EXPANDED_BOOK_PROFILES}"
)

## Reproducible ephemeral checkout

Add two private Colab secrets and enable **Notebook access** for each:

- `GITHUB_TOKEN` (optional): read-only **Contents** access when cloning a private fork.
- `OPENAI_API_KEY`: required only when `ANALYSIS_MODE` is `enhanced`; it is read immediately before the authorized model-assisted stage.

The repository is cloned at one resolved commit and installed from `/content`. It is never cloned into Drive.

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run one shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_colab_secret(name: str) -> str:
    """Read one named secret without exposing provider error text."""

    try:
        from google.colab import userdata

        value = userdata.get(name)
    except Exception:
        raise RuntimeError(
            f"Colab secret {name} is missing or Notebook access is disabled."
        ) from None
    if not value:
        raise RuntimeError(f"Colab secret {name} is empty.")
    return value


def get_optional_github_token() -> str | None:
    """Return an optional read-only token for a private fork."""

    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return None
    return token or None


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield anonymous Git access or temporary authentication for a private fork."""

    environment = os.environ.copy()
    environment.pop("GITHUB_TOKEN", None)
    environment["GIT_TERMINAL_PROMPT"] = "0"
    token = get_optional_github_token()
    if token is None:
        yield environment
        return

    with tempfile.TemporaryDirectory(prefix="cac_git_auth_", dir="/content") as directory:
        helper = Path(directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
            }
        )
        try:
            yield environment
        finally:
            environment.pop("CAC_GIT_TOKEN", None)


def resolve_remote_commit() -> str:
    """Resolve the selected branch to exactly one 40-character Git SHA."""

    if (
        run_command(
            ["git", "check-ref-format", "--branch", REPO_BRANCH],
            capture_output=True,
            check=False,
        ).returncode
        != 0
    ):
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")

    expected_ref = f"refs/heads/{REPO_BRANCH}"
    with authenticated_git_environment() as environment:
        result = run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "ls-remote",
                "--exit-code",
                PUBLIC_REPO_URL,
                expected_ref,
            ],
            env=environment,
            capture_output=True,
        )
    matches = [
        fields[0]
        for line in result.stdout.splitlines()
        if len(fields := line.split()) == 2 and fields[1] == expected_ref
    ]
    if len(matches) != 1 or re.fullmatch(r"[0-9a-f]{40}", matches[0]) is None:
        raise RuntimeError("Selected branch did not resolve to exactly one Git commit.")
    return matches[0]

In [ ]:
import importlib
import inspect

CHECKED_OUT_COMMIT = resolve_remote_commit()
REPO_DIR = Path(
    tempfile.mkdtemp(
        prefix=f"chatgpt-archive-compiler-{CHECKED_OUT_COMMIT[:12]}-",
        dir="/content",
    )
)
with authenticated_git_environment() as environment:
    run_command(
        [
            "git",
            "-c",
            "credential.helper=",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            "--no-tags",
            PUBLIC_REPO_URL,
            str(REPO_DIR),
        ],
        env=environment,
    )
run_command(["git", "checkout", "--detach", CHECKED_OUT_COMMIT], cwd=REPO_DIR)
actual_commit = run_command(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True
).stdout.strip()
if actual_commit != CHECKED_OUT_COMMIT:
    raise RuntimeError("Ephemeral checkout did not resolve to the selected commit.")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "uv==0.11.28",
    ]
)
LOCKED_REQUIREMENTS = Path("/tmp") / f"cac-{CHECKED_OUT_COMMIT[:12]}-requirements.txt"
export_command = [
    "uv",
    "export",
    "--frozen",
    "--no-dev",
    "--no-emit-project",
    "--format",
    "requirements.txt",
    "--output-file",
    str(LOCKED_REQUIREMENTS),
]
for extra in ("notebooks", "pdf", "semantic"):
    export_command.extend(["--extra", extra])
run_command(export_command, cwd=REPO_DIR)
run_command(
    [
        "uv",
        "pip",
        "install",
        "--system",
        "--strict",
        "--requirements",
        str(LOCKED_REQUIREMENTS),
    ]
)
run_command(
    [
        "uv",
        "pip",
        "install",
        "--system",
        "--strict",
        "--no-deps",
        "--editable",
        str(REPO_DIR),
    ]
)
repository_source = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(repository_source))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
semantic_module = importlib.import_module("chatgpt_archive_compiler.semantic")
semantic_book_module = importlib.import_module("chatgpt_archive_compiler.semantic_book")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")

IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
SemanticAtlasOptions = semantic_module.SemanticAtlasOptions
OpenAIEmbeddingProvider = semantic_module.OpenAIEmbeddingProvider
OpenAIStructuredAnalysisProvider = semantic_module.OpenAIStructuredAnalysisProvider
LocalHashingEmbeddingProvider = semantic_module.LocalHashingEmbeddingProvider
LocalHeuristicAnalysisProvider = semantic_module.LocalHeuristicAnalysisProvider
estimate_semantic_run = semantic_module.estimate_semantic_run
build_semantic_atlas = semantic_module.build_semantic_atlas
SemanticBookOptions = semantic_book_module.SemanticBookOptions
BookPaperSize = semantic_book_module.BookPaperSize
compile_semantic_book = semantic_book_module.compile_semantic_book
read_archive_ir = serialization_module.read_archive_ir

imported_from = Path(archive_compiler.__file__).resolve()
if repository_source not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the ephemeral checkout.")
dirty = run_command(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=REPO_DIR,
    capture_output=True,
).stdout
if dirty:
    raise RuntimeError("Package installation unexpectedly changed the Git checkout.")

print(f"Ephemeral checkout: {REPO_DIR}")
print(f"Commit: {CHECKED_OUT_COMMIT}")
print(f"Package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Analytical model

The atlas deliberately keeps four concepts separate:

1. **Subject** — what a conversation discusses.
2. **Project** — the continuing body of work to which it contributes.
3. **Purpose** — what the conversation is doing: exploring, planning, implementing, diagnosing, deciding, reflecting, or revising.
4. **Theme** — a recurring concern or pattern that can connect otherwise different subjects.

Local preprocessing creates privacy-controlled conversation representations. Embeddings build a nearest-neighbor graph; graph communities propose candidate groupings; model-assisted interpretation assigns multi-label profiles, names coherent categories, distinguishes projects from broad interests, and identifies continuations, reversals, unresolved questions, and cross-domain connections. A second global pass reasons over compact category and project profiles rather than blindly concatenating the archive.

Every expensive intermediate result is content-addressed and cached. To keep the global synthesis within a reliable structured-output scale, graph communities are consolidated only when they exceed `MAX_LEAF_CATEGORIES`; each such consolidation is recorded in the review queue. The queue also exposes low-confidence profiles and assignments, isolated conversations, secondary-category ambiguity, and singleton categories instead of concealing uncertainty.

In [ ]:
from collections.abc import Callable
from typing import Any


def construct_supported(model_type: type[Any], values: Mapping[str, Any]) -> Any:
    """Construct a package option/provider object using its declared fields only."""

    model_fields = getattr(model_type, "model_fields", None)
    if isinstance(model_fields, Mapping):
        accepted = set(model_fields)
    else:
        signature = inspect.signature(model_type)
        if any(
            parameter.kind is inspect.Parameter.VAR_KEYWORD
            for parameter in signature.parameters.values()
        ):
            accepted = set(values)
        else:
            accepted = set(signature.parameters)
    return model_type(**{key: value for key, value in values.items() if key in accepted})


def call_supported(function: Callable[..., Any], /, *args: Any, **kwargs: Any) -> Any:
    """Call a public package API while tolerating additive optional parameters."""

    signature = inspect.signature(function)
    if any(
        parameter.kind is inspect.Parameter.VAR_KEYWORD
        for parameter in signature.parameters.values()
    ):
        return function(*args, **kwargs)
    accepted = set(signature.parameters)
    return function(*args, **{key: value for key, value in kwargs.items() if key in accepted})


def semantic_progress(stage: str, completed: int, total: int, cached: int) -> None:
    """Print content-free progress for long resumable semantic stages."""

    allowed_stages = {
        "representations",
        "embeddings",
        "conversation_profiles",
        "semantic_graph",
        "taxonomy",
        "archive_synthesis",
        "artifact_export",
    }
    safe_stage = stage if stage in allowed_stages else "semantic_stage"
    cache_note = f"; {cached:,} cached" if cached else ""
    print(f"[{safe_stage}] {completed:,}/{total:,}{cache_note}")


def make_atlas_options(*, mode: str = ANALYSIS_MODE) -> Any:
    """Build version-compatible semantic options from notebook controls."""

    return construct_supported(
        SemanticAtlasOptions,
        {
            "mode": mode,
            "analysis_mode": mode,
            "analysis_model": ANALYSIS_MODEL,
            "model": ANALYSIS_MODEL,
            "embedding_model": EMBEDDING_MODEL,
            "embedding_dimensions": EMBEDDING_DIMENSIONS,
            "reasoning_effort": REASONING_EFFORT,
            "max_characters_per_conversation": MAX_CHARACTERS_PER_CONVERSATION,
            "embedding_batch_size": EMBEDDING_BATCH_SIZE,
            "analysis_batch_size": ANALYSIS_BATCH_SIZE,
            "max_neighbors": NEAREST_NEIGHBORS,
            "nearest_neighbors": NEAREST_NEIGHBORS,
            "min_similarity": MIN_SIMILARITY,
            "max_leaf_categories": MAX_LEAF_CATEGORIES,
            "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
            "cache_enabled": True,
            "include_reasoning": False,
            "max_conversations": MAX_CONVERSATIONS or None,
        },
    )


SAFE_AGGREGATE_FIELDS = {
    "conversation_count",
    "selected_conversation_count",
    "message_count",
    "character_count",
    "original_character_count",
    "approximate_input_tokens",
    "embedding_item_count",
    "analysis_item_count",
    "truncated_conversation_count",
    "estimated_input_tokens",
    "estimated_output_tokens",
    "estimated_embedding_tokens",
    "estimated_total_tokens",
    "estimated_request_count",
    "estimated_batch_count",
    "estimated_cost_usd",
    "category_count",
    "subcategory_count",
    "theme_count",
    "project_count",
    "edge_count",
    "review_item_count",
    "cached_item_count",
    "computed_item_count",
    "unclassified_conversation_count",
}


def safe_aggregate_summary(value: Any) -> dict[str, int | float | str | None]:
    """Return only explicitly approved, non-content scalar fields."""

    if hasattr(value, "model_dump"):
        raw = value.model_dump(mode="json")
    elif isinstance(value, Mapping):
        raw = dict(value)
    else:
        raw = vars(value) if hasattr(value, "__dict__") else {}
    return {
        key: item
        for key, item in raw.items()
        if key in SAFE_AGGREGATE_FIELDS and (item is None or isinstance(item, (int, float, str)))
    }

## Synthetic full-pipeline validation — no API calls

This test creates two deliberately related conversations in temporary Colab storage, ingests them through the same Archive IR path used by the real export, builds a complete atlas with deterministic local test providers, and renders a small HTML book. It verifies planning, categorization, artifact serialization, and publication without reading an API key or making a network request.

In [ ]:
import json
import zipfile


def synthetic_conversation(
    conversation_id: str, title: str, timestamp: int, question: str, answer: str
) -> dict[str, object]:
    """Create one minimal export-shaped conversation for package validation."""

    return {
        "id": conversation_id,
        "title": title,
        "create_time": timestamp,
        "update_time": timestamp + 60,
        "current_node": f"{conversation_id}-assistant",
        "mapping": {
            f"{conversation_id}-root": {
                "id": f"{conversation_id}-root",
                "parent": None,
                "message": None,
            },
            f"{conversation_id}-user": {
                "id": f"{conversation_id}-user",
                "parent": f"{conversation_id}-root",
                "message": {
                    "id": f"{conversation_id}-message-user",
                    "author": {"role": "user"},
                    "create_time": timestamp,
                    "content": {"content_type": "text", "parts": [question]},
                    "metadata": {},
                },
            },
            f"{conversation_id}-assistant": {
                "id": f"{conversation_id}-assistant",
                "parent": f"{conversation_id}-user",
                "message": {
                    "id": f"{conversation_id}-message-assistant",
                    "author": {"role": "assistant"},
                    "create_time": timestamp + 60,
                    "content": {"content_type": "text", "parts": [answer]},
                    "metadata": {},
                },
            },
        },
    }


synthetic_payload = [
    synthetic_conversation(
        "semantic-one",
        "Synthetic representation-learning study",
        1_735_689_600,
        "Design a reproducible representation-learning comparison.",
        "Define datasets, held-out validation, metrics, and provenance.",
    ),
    synthetic_conversation(
        "semantic-two",
        "Synthetic validation follow-up",
        1_738_368_000,
        "How should the earlier comparison be externally validated?",
        "Preserve the split and test transfer on an independent dataset.",
    ),
]

with tempfile.TemporaryDirectory(prefix="cac_semantic_smoke_", dir="/content") as directory:
    temporary_path = Path(directory)
    synthetic_zip = temporary_path / "synthetic-export.zip"
    with zipfile.ZipFile(synthetic_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive_zip:
        archive_zip.writestr("conversations.json", json.dumps(synthetic_payload, allow_nan=False))
    synthetic_archive = ingest_export_zip(
        synthetic_zip, limits=IngestLimits(), schema_mode=SchemaMode.STRICT
    )
    synthetic_options = make_atlas_options(mode="local")
    synthetic_estimate = call_supported(
        estimate_semantic_run,
        synthetic_archive,
        options=synthetic_options,
    )
    synthetic_summary = safe_aggregate_summary(synthetic_estimate)
    estimated_conversations = synthetic_summary.get(
        "selected_conversation_count", synthetic_summary.get("conversation_count")
    )
    if estimated_conversations is not None:
        assert estimated_conversations == 2
    synthetic_result = call_supported(
        build_semantic_atlas,
        synthetic_archive,
        temporary_path / "atlas",
        embedding_provider=construct_supported(LocalHashingEmbeddingProvider, {}),
        analysis_provider=construct_supported(LocalHeuristicAnalysisProvider, {}),
        options=synthetic_options,
    )
    synthetic_book_options = construct_supported(
        SemanticBookOptions,
        {
            "title": "Synthetic Semantic Atlas",
            "paper_size": BookPaperSize.TRADE,
            "include_transcripts": False,
            "render_pdf": True,
        },
    )
    synthetic_book_result = call_supported(
        compile_semantic_book,
        synthetic_archive,
        synthetic_result.atlas,
        temporary_path / "book",
        options=synthetic_book_options,
    )
    assert synthetic_result.catalog_path.is_file()
    assert synthetic_result.graph_path.is_file()
    assert synthetic_result.taxonomy_path.is_file()
    assert synthetic_result.manifest_path.is_file()
    assert synthetic_result.atlas_html_path.is_file()
    assert synthetic_book_result.html_path.is_file()
    assert synthetic_book_result.pdf_path is not None
    assert synthetic_book_result.pdf_path.read_bytes().startswith(b"%PDF")
    assert synthetic_book_result.manifest_path.is_file()

print("Synthetic Archive-IR-to-atlas-to-book validation passed without API access.")
for key, value in sorted(synthetic_summary.items()):
    print(f"{key}: {value}")

## Real archive preflight — local and resumable

Leave `ARCHIVE_IR_PATH` blank to select the newest successful `archive.ir.json` produced by notebook 01, or supply an exact path to override discovery. A successful candidate must share its directory with `compilation_manifest.json`; ambiguous equally recent candidates are refused. The preflight reads the chosen IR locally, constructs privacy-controlled representations, and reports aggregate token/request estimates. It does **not** retrieve `OPENAI_API_KEY`, create an external provider, or send data to any API.

If a runtime disconnects, set `RESUME_OUTPUT_DIRECTORY` to the exact prior semantic run directory. Its input checksum, repository commit, and analysis configuration must match; otherwise the notebook refuses to mix artifacts. Preparing a new run creates one timestamp-and-commit-stamped directory under `outputs/semantic`. The displayed API cost is a planning estimate: high-effort reasoning tokens are variable and are not a guaranteed component of that estimate.

In [ ]:
import hashlib
from datetime import UTC, datetime

PREPARE_REAL_ANALYSIS = False  # @param {type:"boolean"}
ARCHIVE_IR_PATH = ""  # @param {type:"string"}
RESUME_OUTPUT_DIRECTORY = ""  # @param {type:"string"}
COLAB_PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_COLAB_ACKNOWLEDGEMENT = "I UNDERSTAND THIS READS MY ARCHIVE IN GOOGLE COLAB"


def sha256_file(path: Path) -> str:
    """Hash one file incrementally."""

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


PREPARED_RUN = None
if not PREPARE_REAL_ANALYSIS:
    print("Real semantic preflight remains disabled.")
else:
    if COLAB_PRIVACY_ACKNOWLEDGEMENT != REQUIRED_COLAB_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact Colab privacy acknowledgment is required.")
    if ARCHIVE_IR_PATH.strip():
        archive_ir_path = Path(ARCHIVE_IR_PATH).expanduser().resolve()
    else:
        compiled_output_root = DRIVE_PROJECT_DIR / "outputs" / "compiled"
        candidates = sorted(
            path.resolve()
            for path in compiled_output_root.glob("*/archive.ir.json")
            if (path.parent / "compilation_manifest.json").is_file()
        )
        if not candidates:
            raise RuntimeError(
                "No successful compiled archive was found; provide ARCHIVE_IR_PATH explicitly."
            )
        newest_mtime_ns = max(path.stat().st_mtime_ns for path in candidates)
        newest_candidates = [
            path for path in candidates if path.stat().st_mtime_ns == newest_mtime_ns
        ]
        if len(newest_candidates) != 1:
            raise RuntimeError(
                "Multiple equally recent compiled archives were found; provide "
                "ARCHIVE_IR_PATH explicitly."
            )
        archive_ir_path = newest_candidates[0]
        print(f"Auto-selected Archive IR: {archive_ir_path}")
    if not archive_ir_path.is_file() or archive_ir_path.name != "archive.ir.json":
        raise RuntimeError("ARCHIVE_IR_PATH must identify an existing archive.ir.json file.")

    output_root_resolved = OUTPUT_ROOT.resolve()
    if RESUME_OUTPUT_DIRECTORY.strip():
        semantic_output_directory = Path(RESUME_OUTPUT_DIRECTORY).expanduser().resolve()
        if not semantic_output_directory.is_dir():
            raise RuntimeError("RESUME_OUTPUT_DIRECTORY must identify an existing directory.")
        if not semantic_output_directory.is_relative_to(output_root_resolved):
            raise RuntimeError("Resume directory must remain inside the semantic output root.")
    else:
        run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
        semantic_output_directory = OUTPUT_ROOT / f"{run_id}-{CHECKED_OUT_COMMIT[:12]}"
        semantic_output_directory.mkdir(parents=True, exist_ok=False)

    archive_sha256 = sha256_file(archive_ir_path)
    atlas_options = make_atlas_options()
    plan_identity = {
        "archive_sha256": archive_sha256,
        "repository_commit": CHECKED_OUT_COMMIT,
        "analysis_mode": ANALYSIS_MODE,
        "analysis_model": ANALYSIS_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimensions": EMBEDDING_DIMENSIONS,
        "reasoning_effort": REASONING_EFFORT,
        "synthesis_reasoning_effort": SYNTHESIS_REASONING_EFFORT,
        "profile_max_output_tokens": PROFILE_MAX_OUTPUT_TOKENS,
        "taxonomy_max_output_tokens": TAXONOMY_MAX_OUTPUT_TOKENS,
        "synthesis_max_output_tokens": SYNTHESIS_MAX_OUTPUT_TOKENS,
        "max_characters_per_conversation": MAX_CHARACTERS_PER_CONVERSATION,
        "embedding_batch_size": EMBEDDING_BATCH_SIZE,
        "analysis_batch_size": ANALYSIS_BATCH_SIZE,
        "nearest_neighbors": NEAREST_NEIGHBORS,
        "min_similarity": MIN_SIMILARITY,
        "max_leaf_categories": MAX_LEAF_CATEGORIES,
        "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        "max_conversations": MAX_CONVERSATIONS or None,
    }
    plan_identity_path = semantic_output_directory / "run_identity.json"
    if plan_identity_path.exists():
        existing_identity = json.loads(plan_identity_path.read_text(encoding="utf-8"))
        if existing_identity != plan_identity:
            raise RuntimeError(
                "Resume identity does not match this archive, commit, or configuration."
            )
    else:
        plan_identity_path.write_text(
            json.dumps(plan_identity, indent=2, sort_keys=True, allow_nan=False) + "\n",
            encoding="utf-8",
        )

    try:
        real_archive = read_archive_ir(archive_ir_path)
        real_estimate = call_supported(estimate_semantic_run, real_archive, options=atlas_options)
    except Exception as exception:
        raise RuntimeError(
            f"Semantic preflight failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed."
        ) from None

    estimate_summary = safe_aggregate_summary(real_estimate)
    cost_planning_summary = {}
    if ANALYSIS_MODE == "enhanced":
        planned_conversations = int(estimate_summary.get("conversation_count", 0) or 0)
        planned_source_tokens = int(estimate_summary.get("approximate_input_tokens", 0) or 0)
        planned_analysis_input_tokens = (
            planned_source_tokens
            + planned_conversations * ESTIMATED_META_INPUT_TOKENS_PER_CONVERSATION
            + ESTIMATED_GLOBAL_OUTPUT_TOKENS
        )
        planned_analysis_output_tokens = (
            planned_conversations * ESTIMATED_PROFILE_OUTPUT_TOKENS + ESTIMATED_GLOBAL_OUTPUT_TOKENS
        )
        planned_profile_batches = (
            planned_conversations + ANALYSIS_BATCH_SIZE - 1
        ) // ANALYSIS_BATCH_SIZE
        configured_output_token_ceiling = (
            planned_profile_batches * PROFILE_MAX_OUTPUT_TOKENS
            + TAXONOMY_MAX_OUTPUT_TOKENS
            + SYNTHESIS_MAX_OUTPUT_TOKENS
        )
        planned_embedding_cost = (
            planned_source_tokens / 1_000_000 * EMBEDDING_USD_PER_MILLION_INPUT_TOKENS
        )
        planned_analysis_cost = (
            planned_analysis_input_tokens / 1_000_000 * ANALYSIS_USD_PER_MILLION_INPUT_TOKENS
            + planned_analysis_output_tokens / 1_000_000 * ANALYSIS_USD_PER_MILLION_OUTPUT_TOKENS
        )
        cost_planning_summary = {
            "price_snapshot_date": PRICE_SNAPSHOT_DATE,
            "planned_embedding_input_tokens": planned_source_tokens,
            "planned_analysis_input_tokens": planned_analysis_input_tokens,
            "planned_analysis_output_tokens": planned_analysis_output_tokens,
            "configured_output_token_ceiling": configured_output_token_ceiling,
            "configured_output_token_cost_ceiling_usd": round(
                configured_output_token_ceiling
                / 1_000_000
                * ANALYSIS_USD_PER_MILLION_OUTPUT_TOKENS,
                2,
            ),
            "estimated_embedding_cost_usd": round(planned_embedding_cost, 2),
            "estimated_analysis_cost_usd": round(planned_analysis_cost, 2),
            "estimated_total_cost_usd": round(planned_embedding_cost + planned_analysis_cost, 2),
        }
    (semantic_output_directory / "preflight_estimate.json").write_text(
        json.dumps(
            {"semantic_run": estimate_summary, "cost_planning": cost_planning_summary},
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )
    PREPARED_RUN = {
        "archive": real_archive,
        "archive_sha256": archive_sha256,
        "atlas_options": atlas_options,
        "output_directory": semantic_output_directory,
        "estimate_summary": estimate_summary,
    }

    print("Semantic preflight completed without external API access.")
    for key, value in sorted(estimate_summary.items()):
        print(f"{key}: {value}")
    for key, value in sorted(cost_planning_summary.items()):
        print(f"cost_planning.{key}: {value}")
    if cost_planning_summary:
        print("Cost is a planning estimate, not a quote or spending cap.")
    print(f"Prepared output directory: {semantic_output_directory}")
    print("Review the estimates above before enabling the model-assisted cell.")

## Authorized atlas construction and book rendering

Enhanced mode transmits each bounded title, date, visible-path user/assistant text, and an opaque join key to the configured OpenAI models. It excludes source IDs and paths, warning locations, reasoning traces, system/tool messages, alternate branches, and attachment contents or locations. Responses requests set `store=False`, which prevents application-state storage, but default abuse-monitoring logs may retain prompts and responses for up to 30 days. API data is not used to train OpenAI models unless the account explicitly opts in. The raw archive and generated atlas remain in your Drive, and provider requests are made only after the exact external-processing acknowledgment below.

The renderer treats the taxonomy as editorial structure rather than a decorative index. Top-level domains become parts; substantial categories become chapters; narrower themes become sections; project timelines and high-value cross-links appear as navigational matter. Chronology is preserved within coherent sections and in metadata, while the opening synthesis explains the major connections across the whole archive.

The same cell is safe to rerun against an unchanged prepared directory: completed content-addressed stages are reused. If rendering fails after analysis, rerun it rather than starting a new semantic analysis.

In [ ]:
RUN_REAL_ANALYSIS = False  # @param {type:"boolean"}
EXTERNAL_MODEL_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_EXTERNAL_ACKNOWLEDGEMENT = (
    "I AUTHORIZE OPENAI PROCESSING AND ACCEPT UP TO 30 DAYS OF ABUSE-MONITORING RETENTION"
)

if not RUN_REAL_ANALYSIS:
    print("Real semantic analysis remains disabled.")
else:
    if PREPARED_RUN is None:
        raise RuntimeError("Run the enabled real-archive preflight cell first.")
    if ANALYSIS_MODE == "enhanced" and (
        EXTERNAL_MODEL_ACKNOWLEDGEMENT != REQUIRED_EXTERNAL_ACKNOWLEDGEMENT
    ):
        raise RuntimeError("Exact external-processing acknowledgment is required.")

    embedding_provider = None
    analysis_provider = None
    if ANALYSIS_MODE == "enhanced":
        openai_api_key = get_colab_secret("OPENAI_API_KEY")
        embedding_provider = construct_supported(
            OpenAIEmbeddingProvider,
            {
                "api_key": openai_api_key,
                "model": EMBEDDING_MODEL,
                "dimensions": EMBEDDING_DIMENSIONS,
                "embedding_dimensions": EMBEDDING_DIMENSIONS,
                "request_batch_size": EMBEDDING_BATCH_SIZE,
            },
        )
        analysis_provider = construct_supported(
            OpenAIStructuredAnalysisProvider,
            {
                "api_key": openai_api_key,
                "model": ANALYSIS_MODEL,
                "reasoning_effort": REASONING_EFFORT,
                "synthesis_reasoning_effort": SYNTHESIS_REASONING_EFFORT,
                "profile_max_output_tokens": PROFILE_MAX_OUTPUT_TOKENS,
                "taxonomy_max_output_tokens": TAXONOMY_MAX_OUTPUT_TOKENS,
                "synthesis_max_output_tokens": SYNTHESIS_MAX_OUTPUT_TOKENS,
            },
        )
        del openai_api_key
    else:
        embedding_provider = construct_supported(LocalHashingEmbeddingProvider, {})
        analysis_provider = construct_supported(LocalHeuristicAnalysisProvider, {})

    try:
        atlas_result = call_supported(
            build_semantic_atlas,
            PREPARED_RUN["archive"],
            PREPARED_RUN["output_directory"],
            options=PREPARED_RUN["atlas_options"],
            embedding_provider=embedding_provider,
            analysis_provider=analysis_provider,
            progress_callback=semantic_progress,
        )
    except Exception as exception:
        raise RuntimeError(
            f"Semantic atlas construction failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed. Cached completed stages remain."
        ) from None

    book_options = construct_supported(
        SemanticBookOptions,
        {
            "title": BOOK_TITLE,
            "subtitle": BOOK_SUBTITLE or None,
            "author": BOOK_AUTHOR or None,
            "edition": "First semantic edition",
            "paper_size": BookPaperSize(PAPER_SIZE),
            "render_pdf": RENDER_PDF,
            "include_transcripts": False,
            "include_message_timestamps": False,
            "max_conversations": MAX_CONVERSATIONS or None,
            "max_related_conversations": 8,
            "max_synopsis_entries_per_category": 12,
            "max_expanded_conversations": MAX_EXPANDED_BOOK_PROFILES,
            "max_project_timelines": MAX_PROJECT_TIMELINES,
            "max_timeline_events_per_project": MAX_TIMELINE_EVENTS_PER_PROJECT,
        },
    )
    try:
        book_result = call_supported(
            compile_semantic_book,
            PREPARED_RUN["archive"],
            atlas_result.atlas,
            PREPARED_RUN["output_directory"] / "book",
            options=book_options,
        )
    except Exception as exception:
        raise RuntimeError(
            f"Semantic book rendering failed safely ({type(exception).__name__}); "
            "source-derived exception text was suppressed. Atlas artifacts and cache remain."
        ) from None

    print("Semantic atlas and thematic book completed.")
    for label, result in (("atlas", atlas_result), ("book", book_result)):
        summary = safe_aggregate_summary(result)
        for key, value in sorted(summary.items()):
            print(f"{label}.{key}: {value}")
    for attribute in (
        "catalog_path",
        "graph_path",
        "taxonomy_path",
        "category_profiles_path",
        "project_timelines_path",
        "synthesis_path",
        "review_queue_path",
        "atlas_html_path",
        "html_path",
        "pdf_path",
        "manifest_path",
    ):
        for label, result in (("atlas", atlas_result), ("book", book_result)):
            artifact_path = getattr(result, attribute, None)
            if artifact_path is not None:
                print(f"{label}.{attribute}: {artifact_path}")
    print(f"Output directory: {PREPARED_RUN['output_directory']}")

## Persistent deliverables and review loop

A completed run preserves the semantic catalog, similarity graph, hierarchical taxonomy, category profiles, project timelines, cross-archive synthesis, uncertainty/review queue, provenance manifest, caches, and the HTML/PDF thematic book under:

`MyDrive/ChatGPT Data Export/outputs/semantic/<timestamp>-<commit>/`

The first atlas is intentionally a reviewable draft, not an immutable verdict. Use `review_queue.json` to identify classifications that deserve inspection. A durable rename/merge/split override format is a planned refinement rather than a hidden feature of this first draft. No private archive or generated analysis belongs in GitHub; only package code, tests, documentation, and this orchestration notebook are versioned there.